# Time Series

In this lesson, we'll use the `groupby` idea to answer questions about **when** things happen. Think about a phone's screen-time report: it records activity throughout the day, but shows you a total for each day or week. We already know how to summarize groups. Now our groups will be time intervals. By the end of this lesson, students will be able to:

- Read dates into a **DatetimeIndex** and put observations in chronological order.
- Select a timestamp, a day, or a range of dates using **loc**.
- Explain **resample** as grouping into time intervals, then apply an appropriate summary.
- Explain timestamps, time intervals, and the spacing between observations.
- Distinguish changes over time from recurring calendar patterns and missing observations.

Previously, we grouped earthquakes by place. How about grouping bicycle counts by day?

## A list with a when

Imagine a list of daily screen-time totals: `[30, 70, 110]`. We know the values and their positions, but what dates do they describe? Were they recorded on three consecutive days, or weeks apart?

A **time series** connects each observation to **when it happened**. We can think of it as a familiar list of values with dates attached, or a dictionary whose keys tell us when each value was recorded.

| When | Screen time |
| --- | --- |
| September 2 | 30 minutes |
| September 3 | 70 minutes |
| September 4 | 110 minutes |

The dates let us ask questions that positions alone cannot answer: what happened on September 3? How much time passed between readings? Did screen time increase from one day to the next?

Order matters too. The lists `[30, 70, 110]` and `[110, 70, 30]` have the same sum and mean, but describe opposite changes over time. Sorting observations by their values would lose the chronological story.

**Discuss:** would a list of students' birth dates automatically describe how one quantity changes over time? What would one row need to represent? A date column can be useful without the table being a sequence of observations of the same process.

## What does one row mean?

Before writing code, identify the time and the measurement in each row.

| Familiar example | What the time tells us | What the value measures |
| --- | --- | --- |
| A weather reading | When the thermometer was checked | Temperature at that moment |
| A screen-time report | Which day the report covers | Minutes used during that day |
| An earthquake record | When the event happened | Magnitude of that event |

A **timestamp** identifies a particular date and time, like `2024-09-02 08:00:00`. A **time interval** is a span, like the hour from 8:00 to 9:00. A timestamp may label a measurement taken at a moment or an interval's total; we need to read the dataset's description to know which.

Measurements taken every hour or every day have a regular **frequency**. Earthquakes arrive at irregular times instead. Both can be time series. Two neighboring rows are not necessarily an hour, a day, or even the same amount of time apart.

**Check your understanding:** if two messages were sent at 8:00 and 8:45, are they 1 minute apart because they occupy neighboring rows? Which information would let us find the actual gap?

## From familiar groups to time groups

Recall the three steps in `earthquakes.groupby("name")["magnitude"].max()`: split rows into groups, select a column, and summarize each group. For bicycle counts, we can use exactly that reasoning.

| Question | Groups | Column | Summary |
| --- | --- | --- | --- |
| Largest earthquake in each place? | Place name | Magnitude | `max()` |
| Total bicycle crossings each day? | Calendar day | Crossings | `sum()` |
| Average hourly traffic each day? | Calendar day | Crossings | `mean()` |

To help visualize our work, let's start with six **made-up hourly observations**. Each row counts crossings during the hour starting at `Date`; `East` and `West` are the two sides of a bridge. These are only two hours per day, so their sums describe the hours shown, not a complete day's traffic.

In [1]:
import io
import pandas as pd

csv = """Date,East,West
2024-09-02 08:00:00,4,6
2024-09-02 09:00:00,8,12
2024-09-03 08:00:00,12,18
2024-09-03 09:00:00,16,24
2024-09-04 08:00:00,20,30
2024-09-04 09:00:00,24,36
"""
readings = pd.read_csv(io.StringIO(csv))
readings["Total"] = readings["East"] + readings["West"]
readings

,Date,East,West,Total
0,2024-09-02 08:00:00,4,6,10
1,2024-09-02 09:00:00,8,12,20
2,2024-09-03 08:00:00,12,18,30
3,2024-09-03 09:00:00,16,24,40
4,2024-09-04 08:00:00,20,30,50
5,2024-09-04 09:00:00,24,36,60


Let's first solve this using a dictionary, as we did in the previous lecture. The first ten characters in each date string give us the year, month, and day.

In [2]:
crossings_by_day = {}
for reading in readings.to_dict("records"):
    day = reading["Date"][:10]
    if day not in crossings_by_day:
        crossings_by_day[day] = 0
    crossings_by_day[day] += reading["Total"]

crossings_by_day

{'2024-09-02': 30, '2024-09-03': 70, '2024-09-04': 110}

Explain to a neighbor: which line chooses the group, and which line summarizes it? We should get totals of **30, 70, and 110**. String slicing works for this particular date format, but we'd like pandas to understand the dates themselves.

## Give the index a clock

A **time series** records observations over time: hourly bicycle counts, daily temperatures, or the sequence of earthquakes. Observations do not have to be evenly spaced. In pandas, a convenient way to work with time is to put parsed dates in the index.

We already used `index_col` to choose row labels. Add `parse_dates` to read those labels as dates rather than ordinary strings.

In [3]:
bikes = pd.read_csv(io.StringIO(csv), index_col="Date", parse_dates=["Date"])
bikes = bikes.sort_index()
bikes["Total"] = bikes["East"] + bikes["West"]
bikes

,East,West,Total
Date,,,
2024-09-02 08:00:00,4,6,10
2024-09-02 09:00:00,8,12,20
2024-09-03 08:00:00,12,18,30
2024-09-03 09:00:00,16,24,40
2024-09-04 08:00:00,20,30,50
2024-09-04 09:00:00,24,36,60


In [4]:
bikes.index

DatetimeIndex(['2024-09-02 08:00:00', '2024-09-02 09:00:00',
               '2024-09-03 08:00:00', '2024-09-03 09:00:00',
               '2024-09-04 08:00:00', '2024-09-04 09:00:00'],
              dtype='datetime64[us]', name='Date', freq=None)

Notice the **DatetimeIndex**. Each label is now a `Timestamp`, pandas' representation of a date and time. `index_col` chooses the labels; `parse_dates` gives those labels their time meaning. Setting the index alone does not convert strings into timestamps.

`sort_index()` arranges the rows from earliest to latest. This makes date slices predictable and lets us compare earlier observations with later ones. Like other pandas operations, it returns a result, so we assign it back to `bikes`.

Think back to dictionary keys and DataFrame indexes: the index answers "which row?" A `DatetimeIndex` also gives pandas the calendar information needed to answer "which day?" or "which month?"

### Dates are values, not just text

We can subtract numbers to find their difference. We can also subtract timestamps to find **elapsed time**. `pd.Timestamp` creates a single timestamp, and the subtraction returns a `Timedelta`, a duration.

In [5]:
start = pd.Timestamp("2024-09-02 09:00:00")
end = pd.Timestamp("2024-09-03 08:00:00")
end - start

Timedelta('0 days 23:00:00')

These observations are on consecutive dates, but only **23 hours** apart. Pandas handles the change of day for us. Subtracting the original date strings would not do this: Python treats them as text.

Use dates with the year first, such as `2024-09-02`, to make examples clear. A value like `09/02/2024` can mean different dates in different date conventions. Parsing tells pandas how to interpret the text; it does not fix an ambiguous dataset description.

## Select dates with loc

We still use `.loc[row, column]`. A complete timestamp selects one observation in this dataset.

In [6]:
bikes.loc["2024-09-02 08:00:00", "Total"]

np.int64(10)

A less specific string selects all the timestamps that match. Predict the output type and values before running each example.

In [7]:
bikes.loc["2024-09-02"]

,East,West,Total
Date,,,
2024-09-02 08:00:00,4,6,10
2024-09-02 09:00:00,8,12,20


In [8]:
bikes.loc["2024-09", "Total"]

Date
2024-09-02 08:00:00    10
2024-09-02 09:00:00    20
2024-09-03 08:00:00    30
2024-09-03 09:00:00    40
2024-09-04 08:00:00    50
2024-09-04 09:00:00    60
Name: Total, dtype: int64

In [9]:
bikes.loc["2024-09-02":"2024-09-03", "Total"]

Date
2024-09-02 08:00:00    10
2024-09-02 09:00:00    20
2024-09-03 08:00:00    30
2024-09-03 09:00:00    40
Name: Total, dtype: int64

The first result is a `DataFrame` containing both hours on September 2. The second is a `Series` containing all six totals in September. The last includes both hours on September 2 **and** both hours on September 3: pandas label slices include the end date. A year string such as `"2024"` works the same way.

Unlike slicing a Python list with `[start:stop]`, the stopping label is included. Unlike slicing a string, these selections use the calendar meaning of the index.

**Check your understanding:** write an expression selecting just the `West` counts for September 3 through September 4. How many values should it contain?

## Resample: groupby for time intervals

Let's use the groupby we already know. `bikes.index.date` supplies a full calendar date for every row; repeated dates become one group.

In [10]:
bikes.groupby(bikes.index.date)["Total"].sum()

2024-09-02     30
2024-09-03     70
2024-09-04    110
Name: Total, dtype: int64

Pandas provides `resample` to make grouping into time intervals convenient. Instead of supplying group labels ourselves, we ask pandas to construct daily groups with `"D"`.

In [11]:
daily = bikes.resample("D")["Total"].sum()
daily

Date
2024-09-02     30
2024-09-03     70
2024-09-04    110
Freq: D, Name: Total, dtype: int64

![Six hourly totals split into three daily groups. September 2 contains 10 and 20, totaling 30; September 3 contains 30 and 40, totaling 70; September 4 contains 50 and 60, totaling 110.](time-series-groups.svg)

What's going on here? Read the expression in three steps, just like `groupby`:

1. `bikes.resample("D")` splits the timeline into daily intervals.
2. `["Total"]` selects the values to summarize within each interval.
3. `.sum()` combines those values, producing one total per day.

**Resample is essentially groupby for time series:** the summary step is familiar; the group boundaries come from a time rule. `resample("D")` by itself describes the groups. We still need an operation such as `sum()`, `mean()`, or `max()` to calculate an answer.

Each daily result is labeled with the start of its day (midnight). September 2's group includes timestamps from September 2 at 00:00 up to, but not including, September 3 at 00:00. Those labels describe intervals; they are not extra observations at midnight.

**Predict:** if we replace `.sum()` with `.mean()`, will September 2's answer be 30 or 15? What question does that answer?

In [12]:
bikes.resample("D")["Total"].mean()

Date
2024-09-02    15.0
2024-09-03    35.0
2024-09-04    55.0
Freq: D, Name: Total, dtype: float64

The result is **15 crossings per observed hour** on September 2. Adding hourly counts gives a total; averaging them gives a typical observed hour. Choosing the time interval and choosing the summary are two separate decisions.

| Time rule | Groups | Example use |
| --- | --- | --- |
| `"h"` | Hours | Hourly readings |
| `"D"` | Calendar days | Daily totals |
| `"W"` | Weeks ending Sunday | Weekly totals |
| `"MS"` | Calendar months, labeled at month start | Monthly averages |
| `"YS"` | Calendar years, labeled at year start | Annual totals |

For `"W"`, the default groups run Monday through Sunday and are labeled by Sunday. Calendar rules have different boundaries; a month is not always 30 days. We'll use `MS` and `YS` for clear start-of-period labels.

## Choose the detail that answers the question

The **granularity** is how much time each row represents. Think of a school calendar: the daily view shows individual classes, while the weekly view gives a broader summary. Resampling changes the time detail of our table.

| Question | Useful time detail | What we need |
| --- | --- | --- |
| Is 8:00 busier than 9:00? | Individual hours | Keep the hourly observations |
| How many crossings were recorded each day? | Days | Add the hourly counts within each day |
| What was a typical recorded hour each day? | Days | Average the hourly counts within each day |

Combining smaller intervals into larger ones is called **downsampling**. It reduces detail: from the daily total 30, we cannot recover that its two hourly counts were 10 and 20. Many different pairs add up to 30.

Selection and resampling also answer different questions. `.loc` keeps the original observations that match a date; `resample` calculates a new summary for each interval. Compare these results.

In [13]:
bikes.loc["2024-09-02", "Total"]

Date
2024-09-02 08:00:00    10
2024-09-02 09:00:00    20
Name: Total, dtype: int64

In [14]:
bikes.resample("D")["Total"].sum().loc["2024-09-02"]

np.int64(30)

The first result keeps **two hourly values**, 10 and 20. The second is **one daily summary**, 30. Neither operation changes `bikes` unless we assign its result back.

**Predict:** could we add hourly temperatures to find the temperature for a day? Counts of separate crossings add up meaningfully; temperatures usually call for an average, minimum, or maximum instead. A time rule chooses the groups, but the meaning of the measurement determines the summary.

### Compare consecutive summaries

With a list, we might subtract `values[i - 1]` from `values[i]` to find how much a value changed. A pandas `Series` has `.diff()` for this same comparison with the previous row.

In [15]:
daily.diff()

Date
2024-09-02     NaN
2024-09-03    40.0
2024-09-04    40.0
Freq: D, Name: Total, dtype: float64

The changes are `NaN`, **40**, and **40**. There is no earlier row to compare with the first daily total. After that, the recorded totals rose by 40 each day.

Here, neighboring rows represent consecutive days because we resampled by day. On an irregular series, `.diff()` would still compare neighboring rows, even if they were different amounts of time apart. Always check what the index means before calling a change "per day."

## Same month, different year

Suppose we have four daily totals from two different Januaries. Should January 2023 and January 2024 belong to the same group? It depends on our question.

In [16]:
january = pd.DataFrame(
    {"Total": [10, 20, 100, 200]},
    index=pd.to_datetime(["2023-01-01", "2023-01-02", "2024-01-01", "2024-01-02"]),
)
january

,Total
2023-01-01,10
2023-01-02,20
2024-01-01,100
2024-01-02,200


`index.month` extracts just a number from 1 to 12. Predict how many groups this makes.

In [17]:
january.groupby(january.index.month)["Total"].sum()

1    330
Name: Total, dtype: int64

There is one group labeled `1`, with total **330**. The year was left out of our grouping key. That answers "How many crossings were recorded in January across all years?"

Now ask for a separate result for each calendar month on the timeline.

In [18]:
january.resample("MS")["Total"].sum(min_count=1)

2023-01-01     30.0
2023-02-01      NaN
2023-03-01      NaN
2023-04-01      NaN
2023-05-01      NaN
2023-06-01      NaN
2023-07-01      NaN
2023-08-01      NaN
2023-09-01      NaN
2023-10-01      NaN
2023-11-01      NaN
2023-12-01      NaN
2024-01-01    300.0
Freq: MS, Name: Total, dtype: float64

January 2023 has **30**, while January 2024 has **300**. The months between them appear too, with missing values (`NaN`), because resampling constructs a sequence of time intervals. `min_count=1` asks for at least one non-missing observation before reporting a sum.

| Our question | Appropriate grouping |
| --- | --- |
| How did traffic change month by month over time? | `resample("MS")` keeps each year-month separate |
| What is a typical January compared with a typical July? | Group by `index.month` and use an appropriate average |

The same issue applies to week numbers: week 1 repeats each year. This is a choice of **grouping key**, not a limitation of `groupby`. For example, `groupby([january.index.year, january.index.month])` keeps the years separate too. Resampling handles calendar intervals and their date labels for us.

## Missing observations are not zero

In our small example, imagine the sensor stopped recording for all of September 3. Remove those rows, then ask for a daily result.

In [19]:
with_gap = bikes.loc[bikes.index.day != 3]
with_gap.resample("D")["Total"].sum(min_count=1)

Date
2024-09-02     30.0
2024-09-03      NaN
2024-09-04    110.0
Freq: D, Name: Total, dtype: float64

September 3 is `NaN`, not zero. Zero would mean we measured no crossings; here we have no measurements. Without `min_count=1`, an empty group's sum would be zero and could hide the gap.

Use `isna()` to identify missing values. `dropna()` removes them; `fillna(0)` replaces them with zero. Either choice changes what your data says. Don't fill an unobserved traffic count with zero unless you have evidence that zero is correct.

One recorded hour is enough to satisfy `min_count=1`, so it does **not** guarantee a complete day. A count of valid observations helps us check coverage.

In [20]:
with_gap.resample("D")["Total"].count()

Date
2024-09-02    2
2024-09-03    0
2024-09-04    2
Freq: D, Name: Total, dtype: int64

We get **2, 0, 2** valid readings. Both `mean()` and `sum()` normally skip missing values; even a non-missing summary can be based on incomplete data. For a real hourly dataset, compare coverage with the expected readings before interpreting a daily or weekly result.

### More rows do not mean more measurements

What if we ask for a reading every 30 minutes when our sensor only recorded each hour? That is a finer frequency, often called **upsampling**. Adding time slots cannot tell us what the sensor would have measured in between.

`.asfreq("30min")` places observations on a 30-minute grid. For this example, we first select one day's two readings.

In [21]:
bikes.loc["2024-09-02", "Total"].asfreq("30min")

Date
2024-09-02 08:00:00    10.0
2024-09-02 08:30:00     NaN
2024-09-02 09:00:00    20.0
Freq: 30min, Name: Total, dtype: float64

There are now three rows: 8:00 has **10**, 8:30 is **NaN**, and 9:00 has **20**. The middle slot is a place for a possible measurement, not a newly observed value. Guessing it would require an additional assumption.

Compare this with daily resampling: downsampling combines existing readings into summaries; creating a finer grid exposes times for which we have no reading.

## Ask time questions about a real dataset

Let's now load the actual **Fremont Bridge bicycle counts** used in the [CSE 163 time-series reading](https://cse163.github.io/book/module-3-pandas/lesson-9-time-series/time-series.html). The local [bicycles.csv](bicycles.csv) contains hourly counts from October 2012 through March 2019, with `East` and `West` columns. Run the notebook with this CSV beside it.

First, choose and parse the time index, sort it, and add the two sides. This CSV writes dates like `03/31/2019 11:00:00 PM`; `date_format` tells pandas the exact month/day/year and 12-hour clock format to read. If either side is missing, their sum is missing too.

In [22]:
bicycles = pd.read_csv(
    "bicycles.csv", index_col="Date", parse_dates=["Date"],
    date_format="%m/%d/%Y %I:%M:%S %p",
)
bicycles = bicycles.sort_index()
bicycles["Total"] = bicycles["East"] + bicycles["West"]
bicycles.head()

,East,West,Total
Date,,,
2012-10-03 00:00:00,4.0,9.0,13.0
2012-10-03 01:00:00,4.0,6.0,10.0
2012-10-03 02:00:00,1.0,1.0,2.0
2012-10-03 03:00:00,2.0,3.0,5.0
2012-10-03 04:00:00,6.0,1.0,7.0


Let's ask two questions: "What readings were recorded during March 2017?" and "How many crossings were recorded in each week?" Before running the cells, identify which one selects existing observations and which one groups and summarizes.

In [23]:
bicycles.loc["2017-03", "Total"].head()

Date
2017-03-01 00:00:00    3.0
2017-03-01 01:00:00    4.0
2017-03-01 02:00:00    2.0
2017-03-01 03:00:00    1.0
2017-03-01 04:00:00    6.0
Name: Total, dtype: float64

In [24]:
weekly = bicycles.resample("W")["Total"].sum(min_count=1)
weekly.head()

Date
2012-10-07    14292.0
2012-10-14    16795.0
2012-10-21    15509.0
2012-10-28    13437.0
2012-11-04    12194.0
Freq: W-SUN, Name: Total, dtype: float64

The first cell selects March's hourly readings and displays the first five. The second produces a `Series` with one result for each week and displays its first five results. Notice that the units changed from crossings **per hour** to crossings **per week**. We summarized the observations in every week; we did not just keep one arbitrary hour from each week.

**Discuss:** could we fairly compare the first week's total with a complete week's total? The file begins partway through a week. A calendar group can contain fewer observations at the edge of a dataset, even if none of the recorded values are missing.

## Changes and repeating patterns

We can ask two different kinds of time questions without making a plot.

| Pattern | Familiar example | What we compare |
| --- | --- | --- |
| **Trend:** a longer-term rise or fall | Your weekly reading time gradually increases during a semester | Earlier periods with later periods, keeping them in order |
| **Seasonality:** a pattern that repeats on a calendar or cycle | Screen time tends to be higher on weekends | Matching parts of repeated weeks |

The two can occur together: your reading time could increase over the semester while still being lower every Friday. One unusually busy day alone does not establish either pattern.

For bicycle traffic, keeping separate weeks on the timeline helps us investigate change over time. Grouping by the hour of day instead helps ask about a recurring daily pattern. We already know the groupby syntax; the new piece is extracting the hour from the time index.

In [25]:
bicycles.groupby(bicycles.index.hour)["Total"].mean()

Date
0      11.632644
1       6.113876
2       4.080727
3       2.983551
4       6.557149
5      25.799241
6      90.043442
7     225.537326
8     317.905103
9     185.941350
10     97.166667
11     84.979747
12     89.799578
13     96.025306
14    104.248418
15    132.576550
16    220.194011
17    379.426824
18    261.256010
19    127.668073
20     73.544074
21     47.160270
22     31.377478
23     20.810207
Name: Total, dtype: float64

The group labels are **0 through 23**, not dates. Each group combines the same clock hour across many days and years. Its value is the mean recorded hourly count for that hour of day. This can help us investigate a typical commute pattern, but it cannot tell us whether traffic increased from 2013 to 2018.

**Check your understanding:** which operation keeps a timeline, `resample("D")` or `groupby(bicycles.index.hour)`? Why might weekdays and weekends have different typical hours?

## Practice: earthquake timelines

Let's return to the earthquake data from the previous lectures. This file stores `year`, `month`, and `day` separately. `pd.to_datetime` can combine those three columns into dates. `set_index` then makes the dates our row labels.

Many earthquakes occur on the same day, so these date labels repeat. A `DatetimeIndex` can contain duplicates; resampling includes all rows in each time interval.

In [26]:
earthquakes = pd.read_csv("earthquakes.csv", index_col="id")
earthquakes["Date"] = pd.to_datetime(earthquakes[["year", "month", "day"]])
earthquakes = earthquakes.set_index("Date").sort_index()
earthquakes.head()

,year,month,day,latitude,longitude,name,magnitude
Date,,,,,,,
2016-07-27,2016,7,27,37.672333,-121.619000,California,1.43
2016-07-27,2016,7,27,21.514600,94.572100,Burma,4.90
2016-07-27,2016,7,27,37.576500,-118.859167,California,0.06
2016-07-27,2016,7,27,37.595833,-118.994833,California,0.40
2016-07-27,2016,7,27,39.377500,-119.845000,Nevada,0.30


Select all earthquakes from **August 10 through August 15, 2016**, including both endpoints. How does this compare to selecting from a hierarchical year/month/day index?

Compute the **largest magnitude on each day**. Predict whether the result is a single value, a `Series`, or a `DataFrame`.

Compute the **number of earthquakes in each week**. Use `.size()` to count rows in each group, including rows with a missing magnitude. How would `.count()` on `"magnitude"` differ?

Explain why grouping by `earthquakes.index.day` would combine unrelated dates. Write a resampling expression that keeps each day's events together without mixing months or years.

## Check your understanding

1. How are `groupby("name")["magnitude"].max()` and `resample("D")["magnitude"].max()` alike? What changes about how we choose groups?
2. What does `bicycles.loc["2017-03"]` select? What does `bicycles.resample("MS")["Total"].sum(min_count=1)` calculate instead?
3. A student writes `bicycles.groupby(bicycles.index.month)["Total"].sum()` and calls the result a timeline. What information has been lost?
4. Why might a small weekly total be misleading even when we use `min_count=1`?
5. How is a timestamp different from a duration? Why might two neighboring rows be more than one hour apart?
6. If hourly readings are placed on a 30-minute grid, what do the new missing slots tell us? Can we recover the original hourly values from just daily sums?

For the next lecture, keep this connection in mind: select observations with `.loc`, group them into time intervals with `.resample`, and choose a summary that answers your question.

This lesson adapts the topics in Hunter Schafer's [CSE 163 Lesson 9: Time Series](https://cse163.github.io/book/module-3-pandas/lesson-9-time-series/index.html), using new introductory examples and the site's notebook lecture format. The bicycle CSV is the [snapshot distributed with that reading](https://cse163.github.io/book/_downloads/8cfb27e1060ca269d177c4cca6886ddc/bicycles.csv). For more time rules and boundary options, see the [pandas time-series guide](https://pandas.pydata.org/docs/user_guide/timeseries.html) and [`resample` reference](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html).